In [1]:
import sqlite3

In [2]:
laundry_mat_data = [
    {"product": "Detergent", "dollar_price_per_unit": 4.5, "quantity": 100, "total_cost": 200},
    {"product": "Dryer Sheets Box", "dollar_price_per_unit": 3.5, "quantity": 100, "total_cost": 350},
    {"product": "Washing Machine", "dollar_price_per_unit": 400, "quantity": 25, "total_cost": 10000},
    {"product": "Dryer", "dollar_price_per_unit": 400, "quantity": 25, "total_cost": 10000},
]

In [3]:
conn = sqlite3.connect('./Data/laundry_mat.db')
cursor = conn.cursor()

## **Full data loads**

In [4]:
cursor.execute("DROP TABLE IF EXISTS laundry_mat")

In [5]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS laundry_mat (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        product TEXT,
        dollar_price_per_unit REAL,
        quantity INTEGER,
        total_cost REAL
    )
""")

In [6]:
# Convert your list of dictionaries into a list of tuples
rows = [
    (item['product'], item['dollar_price_per_unit'], item['quantity'], item['total_cost'])
    for item in laundry_mat_data
]

cursor.executemany("""
    INSERT INTO laundry_mat (product, dollar_price_per_unit, quantity, total_cost) 
    VALUES (?, ?, ?, ?)
""", rows)

In [7]:
# 1. Execute the query to select all records from the table
cursor.execute("SELECT * FROM laundry_mat")

# 2. Fetch all the rows
rows = cursor.fetchall()

# 3. Print or process each row
for row in rows:
    print(row)

(1, 'Detergent', 4.5, 100, 200.0)
(2, 'Dryer Sheets Box', 3.5, 100, 350.0)
(3, 'Washing Machine', 400.0, 25, 10000.0)
(4, 'Dryer', 400.0, 25, 10000.0)


## **Incremental data loads**

In [8]:
laundry_mat_data = [
    {"product": "Detergent", "dollar_price_per_unit": 4.5, "quantity": 100, "total_cost": 200},
    {"product": "Dryer Sheets Box", "dollar_price_per_unit": 3.5, "quantity": 100, "total_cost": 350},
    {"product": "Washing Machine", "dollar_price_per_unit": 400, "quantity": 25, "total_cost": 10000},
    {"product": "Dryer", "dollar_price_per_unit": 400, "quantity": 25, "total_cost": 10000},
    {"product": "Mouse", "dollar_price_per_unit": 20, "quantity": 25, "total_cost": 450}
]

### **Option A:**

In [9]:
cursor.execute("DROP TABLE IF EXISTS laundry_mat")

In [10]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS laundry_mat (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        product TEXT UNIQUE,
        dollar_price_per_unit REAL,
        quantity INTEGER,
        total_cost REAL
    )
""")

In [11]:
for record in laundry_mat_data:
    cursor.execute("""
        INSERT OR IGNORE INTO laundry_mat 
        (product, dollar_price_per_unit, quantity, total_cost) 
        VALUES (?, ?, ?, ?)
    """, (
        record["product"],
        record["dollar_price_per_unit"],
        record["quantity"],
        record["total_cost"]
    ))

In [12]:
# 1. Execute the query to select all records from the table
cursor.execute("SELECT * FROM laundry_mat")

# 2. Fetch all the rows
rows = cursor.fetchall()

# 3. Print or process each row
for row in rows:
    print(row)

(1, 'Detergent', 4.5, 100, 200.0)
(2, 'Dryer Sheets Box', 3.5, 100, 350.0)
(3, 'Washing Machine', 400.0, 25, 10000.0)
(4, 'Dryer', 400.0, 25, 10000.0)
(5, 'Mouse', 20.0, 25, 450.0)


### **Option B:**

In [13]:
cursor.execute("DROP TABLE IF EXISTS laundry_mat")

In [14]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS laundry_mat (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        product TEXT UNIQUE,
        dollar_price_per_unit REAL,
        quantity INTEGER,
        total_cost REAL
    )
""")

In [15]:
for record in laundry_mat_data:
    cursor.execute("""
        INSERT INTO laundry_mat 
        (product, dollar_price_per_unit, quantity, total_cost) 
        VALUES (?, ?, ?, ?)
        ON CONFLICT(product) DO UPDATE SET
            dollar_price_per_unit = excluded.dollar_price_per_unit,
            quantity = excluded.quantity,
            total_cost = excluded.total_cost
    """, (
        record["product"],
        record["dollar_price_per_unit"],
        record["quantity"],
        record["total_cost"]
    ))

In [16]:
# 1. Execute the query to select all records from the table
cursor.execute("SELECT * FROM laundry_mat")

# 2. Fetch all the rows
rows = cursor.fetchall()

# 3. Print or process each row
for row in rows:
    print(row)

(1, 'Detergent', 4.5, 100, 200.0)
(2, 'Dryer Sheets Box', 3.5, 100, 350.0)
(3, 'Washing Machine', 400.0, 25, 10000.0)
(4, 'Dryer', 400.0, 25, 10000.0)
(5, 'Mouse', 20.0, 25, 450.0)
